# Lab: Practice — Multi Flow SDP with Liquid Clustering and Data Quality
Complete each **TODO**, then run the check cell beneath it. Run the setup cell first.

In [ ]:
%run ./Includes/Classroom-Setup.py

In [ ]:
from pyspark.sql import functions as F
from pipeline import MultiFlowPipeline
DA.reset()
pipe = MultiFlowPipeline(spark, DA.paths.working_dir)
pipe.run()
DA.register_tables(pipe)

## Exercise 1 — Add a third append flow
Create a landing folder `source_flow_c`, land a file into it, register it as a new flow in `pipe.flows`, and run Bronze.

*Hint:* `DA.paths.source_flow_c = ...` and then `DA.land_new_files("c")`.

In [ ]:
# TODO


In [ ]:
# CHECK
assert spark.table("bronze_events").select("source_flow").distinct().count() == 3, "flow_c not ingested"
print("Exercise 1 passed")

## Exercise 2 — Add a new expectation
Add an `expect_or_drop` rule called `plausible_amount` with `amount < 400`, land a new file in flow A, and run Silver.

*Hint:* `pipe.dq.expect_or_drop(...)`

In [ ]:
# TODO


In [ ]:
# CHECK
q = spark.table("silver_quarantine")
assert q.filter(F.array_contains("failed_expectations", "plausible_amount")).count() > 0
print("Exercise 2 passed")
# Think about it: rows with amount >= 400 from EARLIER batches are still in silver_events. Why?

## Exercise 3 — Data skipping with Liquid Clustering
1. Run `OPTIMIZE silver_events`.
2. Query `silver_events WHERE status = 'ERROR'`.
3. Open the Spark UI (link printed by the setup cell), go to **SQL / DataFrame**, and compare *number of files read* against `numFiles` in `DESCRIBE DETAIL`.

In [ ]:
%%sql
-- TODO


## Exercise 4 — Rejection-rate Gold table
Using `silver_events` and `silver_quarantine`, build a table `gold_rejection_rate` with columns
`source_flow, accepted, rejected, rejection_pct`, and save it as a Delta table in the `sdp_demo` schema.

In [ ]:
# TODO


In [ ]:
# CHECK
r = spark.table("gold_rejection_rate")
assert set(r.columns) >= {"source_flow", "accepted", "rejected", "rejection_pct"}
display(r)
print("Exercise 4 passed")

## Exercise 5 — Recover from `expect_or_fail`
Land a file with `null_event_id=True` and confirm Silver fails. Then think about it: the bad file is now in Bronze and the Silver checkpoint keeps hitting it.
What are your options? (Fix the rule, filter upstream, or a full refresh.) Implement a **full refresh of Silver**: delete the Silver table and its checkpoint, relax the rule, then re-run.

In [ ]:
# TODO


## Clean up

In [ ]:
DA.reset()